# 🎙️ Entraînement du wake word « Jarvis » (prononciation française)

Ce notebook entraîne un modèle **openWakeWord** custom pour détecter « jarvis » prononcé **à la française** ([ʒaʁvis]), à déposer ensuite dans `N:\JARVIS\core\jarvis_fr.onnx`.

**Recette :**
1. Génération de ~7 000 échantillons synthétiques de « jarvis » avec 5 voix TTS françaises (Piper), vitesses et intonations variées
2. Génération de ~3 500 échantillons *pièges* (« parvis », « gervais », « service », « jarret »…) pour apprendre au modèle à les rejeter
3. Augmentation : bruits de fond (musique FMA) + réverbérations de pièces réelles (MIT RIR)
4. Entraînement contre 2 000 h de features audio négatives pré-calculées

**⚙️ Avant de lancer : `Exécution` → `Modifier le type d'exécution` → **GPU (T4)**.

**Durée totale : ~1 h** (génération ~20 min, téléchargements ~10 min, entraînement ~30 min).

> Basé sur le pipeline officiel `openwakeword/examples/custom_model.yml`. Si une cellule échoue (les versions des dépendances bougent), copie l'erreur à Claude qui corrigera le notebook.

In [ ]:
# ── 1. Installation des dépendances ─────────────────────────────────────────
!git clone https://github.com/dscripka/openwakeword
%pip install -e ./openwakeword
%pip install piper-tts soundfile scipy tqdm pyyaml
%pip install mutagen torchinfo torchmetrics speechbrain audiomentations torch-audiomentations acoustics datasets
print("\n✔ Installation terminée")

In [ ]:
# ── 2. Téléchargement des voix TTS françaises (Piper) ───────────────────────
import os, urllib.request

VOICES = {
    "fr_FR-siwis-medium":  "fr/fr_FR/siwis/medium/fr_FR-siwis-medium",
    "fr_FR-tom-medium":    "fr/fr_FR/tom/medium/fr_FR-tom-medium",
    "fr_FR-upmc-medium":   "fr/fr_FR/upmc/medium/fr_FR-upmc-medium",   # 2 locuteurs
    "fr_FR-gilles-low":    "fr/fr_FR/gilles/low/fr_FR-gilles-low",
    "fr_FR-mls-medium":    "fr/fr_FR/mls/medium/fr_FR-mls-medium",     # ~125 locuteurs
}
base = "https://huggingface.co/rhasspy/piper-voices/resolve/main/"
os.makedirs("voix_fr", exist_ok=True)
for name, path in VOICES.items():
    try:
        for ext in (".onnx", ".onnx.json"):
            dest = f"voix_fr/{name}{ext}"
            if not os.path.exists(dest):
                urllib.request.urlretrieve(base + path + ext, dest)
        print("✔", name)
    except Exception as e:
        print("✗ ignorée :", name, "—", e)   # au moins siwis/tom/upmc doivent passer

In [ ]:
# ── 3. Génération des échantillons français ─────────────────────────────────
import io, json, wave, random
import numpy as np
from pathlib import Path
from scipy.signal import resample_poly
from tqdm import tqdm
from piper import PiperVoice

OUT = Path("mon_modele")   # = output_dir de l'entraînement
for d in ["positive_train", "positive_test", "negative_train", "negative_test"]:
    (OUT / d).mkdir(parents=True, exist_ok=True)

# Orthographes qui forcent la prononciation française du S final
# (certaines voix TTS liraient « jarvis » comme « jarvi »)
POSITIFS = ["jarvis", "jarviss", "jarvisse"]

# Mots phonétiquement proches que le modèle doit apprendre à REJETER
NEGATIFS = ["parvis", "gervais", "gervaise", "service", "jarret", "jardin",
            "jadis", "tandis", "avis", "mavis", "travis", "clavier",
            "vernis", "tapis", "marquis", "j'arrive", "esprit de service"]

voices = []
for f in sorted(Path("voix_fr").glob("*.onnx")):
    v = PiperVoice.load(str(f))
    n_spk = json.load(open(str(f) + ".json")).get("num_speakers", 1) or 1
    voices.append((v, n_spk, f.stem))
print("Voix chargées :", [(n, s) for _, s, n in voices])

def synth(voice, text, speaker, length_scale, noise_scale):
    # API piper-tts >= 1.3 : SynthesisConfig + generateur de chunks
    try:
        from piper import SynthesisConfig
        try:
            scfg = SynthesisConfig(speaker_id=speaker, length_scale=length_scale, noise_scale=noise_scale)
        except TypeError:
            scfg = SynthesisConfig(length_scale=length_scale, noise_scale=noise_scale)
        chunks = list(voice.synthesize(text, syn_config=scfg))
        if chunks:
            audio = np.frombuffer(b"".join(c.audio_int16_bytes for c in chunks), dtype=np.int16)
            sr = getattr(chunks[0], "sample_rate", None) or voice.config.sample_rate
        else:
            audio = np.zeros(1600, dtype=np.int16); sr = 16000
    except ImportError:
        # API piper-tts <= 1.2 : ecriture directe dans un fichier wave
        buf = io.BytesIO()
        with wave.open(buf, "wb") as w:
            voice.synthesize(text, w, speaker_id=speaker,
                             length_scale=length_scale, noise_scale=noise_scale)
        buf.seek(0)
        with wave.open(buf, "rb") as w:
            sr = w.getframerate()
            audio = np.frombuffer(w.readframes(w.getnframes()), dtype=np.int16)
    if sr != 16000:
        audio = resample_poly(audio.astype(np.float32), 16000, sr).astype(np.int16)
    return audio

def generer(textes, dossier, n_total):
    for i in tqdm(range(n_total), desc=dossier.name):
        voice, n_spk, _ = random.choice(voices)
        audio = synth(voice,
                      random.choice(textes),
                      speaker=(random.randrange(n_spk) if n_spk > 1 else None),
                      length_scale=random.uniform(0.7, 1.4),   # débit lent→rapide
                      noise_scale=random.uniform(0.3, 0.9))    # intonation variable
        with wave.open(str(dossier / f"{i:06d}.wav"), "wb") as w:
            w.setnchannels(1); w.setsampwidth(2); w.setframerate(16000)
            w.writeframes(audio.tobytes())

generer(POSITIFS, OUT / "positive_train", 6000)
generer(POSITIFS, OUT / "positive_test", 1000)
generer(NEGATIFS, OUT / "negative_train", 3000)
generer(NEGATIFS, OUT / "negative_test", 500)
print("\n✔ Échantillons générés")

In [ ]:
# ── 4. Données d'augmentation et features négatives ─────────────────────────
import os, urllib.request
import numpy as np, scipy.io.wavfile
from tqdm import tqdm
from datasets import load_dataset, Audio

# Réverbérations de pièces réelles (MIT RIR)
os.makedirs("./mit_rirs", exist_ok=True)
rir = load_dataset("davidscripka/MIT_environmental_impulse_responses",
                   split="train", streaming=True)
for row in tqdm(rir.cast_column("audio", Audio()), desc="RIR"):
    name = row["audio"]["path"].split("/")[-1]
    scipy.io.wavfile.write("./mit_rirs/" + name, 16000,
                           (row["audio"]["array"] * 32767).astype(np.int16))

# Bruits de fond : 1000 extraits musicaux (FMA)
os.makedirs("./fma_16k", exist_ok=True)
fma = load_dataset("rudraml/fma", name="small", split="train", streaming=True)
fma = iter(fma.cast_column("audio", Audio(sampling_rate=16000)))
for i in tqdm(range(1000), desc="FMA"):
    row = next(fma)
    scipy.io.wavfile.write(f"./fma_16k/{i:05d}.wav", 16000,
                           (row["audio"]["array"] * 32767).astype(np.int16))

# Features négatives pré-calculées (~2 Go : 2000 h d'audio divers)
base = "https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/"
for f in ["openwakeword_features_ACAV100M_2000_hrs_16bit.npy",
          "validation_set_features.npy"]:
    if not os.path.exists(f):
        print("Téléchargement", f, "…")
        urllib.request.urlretrieve(base + f, f)
print("\n✔ Données prêtes")

In [ ]:
# ── 5. Configuration de l'entraînement ──────────────────────────────────────
import yaml

cfg = yaml.load(open("openwakeword/examples/custom_model.yml"), Loader=yaml.FullLoader)
cfg["model_name"] = "jarvis_fr"
cfg["target_phrase"] = ["jarvis"]
cfg["output_dir"] = "./mon_modele"
cfg["n_samples"] = 6000
cfg["n_samples_val"] = 1000
cfg["rir_paths"] = ["./mit_rirs"]
cfg["background_paths"] = ["./fma_16k"]
cfg["false_positive_validation_data_path"] = "validation_set_features.npy"
cfg["feature_data_files"] = {"ACAV100M_sample": "openwakeword_features_ACAV100M_2000_hrs_16bit.npy"}
cfg["custom_negative_phrases"] = []   # déjà générés en français à l'étape 3
cfg["steps"] = 30000

yaml.dump(cfg, open("jarvis_fr.yml", "w"))
print(yaml.dump(cfg))

In [ ]:
# ── 6. Augmentation (bruit + réverbération) puis entraînement ───────────────
# (~30 min sur GPU T4 — la barre de progression avance par paliers, c'est normal)
!python openwakeword/openwakeword/train.py --training_config jarvis_fr.yml --augment_clips
!python openwakeword/openwakeword/train.py --training_config jarvis_fr.yml --train_model

In [ ]:
# ── 7. Récupération du modèle ───────────────────────────────────────────────
import glob
candidats = glob.glob("./mon_modele/**/*.onnx", recursive=True)
print("Modèles produits :", candidats)

from google.colab import files
for c in candidats:
    if "jarvis_fr" in c:
        files.download(c)
        print("⬇ Téléchargé :", c)

## 📦 Installation dans JARVIS

1. Copie le fichier téléchargé dans **`N:\JARVIS\core\jarvis_fr.onnx`** (renomme-le si besoin)
2. Relance JARVIS — `core/wakeword.py` détecte automatiquement ce fichier et bascule dessus.
   Tu verras au démarrage : `[WAKEWORD] Modèle custom français détecté : jarvis_fr.onnx`
3. Dis « Jarvis » et observe les scores `[WAKEWORD]` dans la console. Ajuste au besoin
   `"wakeword_threshold"` dans `jarvis_config.json` (commence à `0.5` avec le modèle custom,
   il est bien plus confiant que hey_jarvis sur la prononciation française).
4. Pour revenir au modèle « hey jarvis » : supprime ou renomme `core/jarvis_fr.onnx`.